In [18]:
import os
import shutil

# Путь к файлам (уже не архив, а папка с файлами)
source_dir = "/kaggle/input/datasets/kolyagudkov/result-scale2/results_scale_2/results_scale"
output_dir = "/kaggle/working/results_scale_2"

# Копируем все файлы
os.makedirs(output_dir, exist_ok=True)

for item in os.listdir(source_dir):
    s = os.path.join(source_dir, item)
    d = os.path.join(output_dir, item)
    if os.path.isfile(s):
        shutil.copy2(s, d)
    else:
        shutil.copytree(s, d, dirs_exist_ok=True)

print(f"Файлы скопированы в {output_dir}")

Файлы скопированы в /kaggle/working/results_scale_2


In [17]:
!ls /kaggle/input/datasets/kolyagudkov/result-scale2/results_scale_2/results_scale

Base		  IP_Face_LoRA_075  IP_Plus_LoRA_05   LoRA_05
IP_Face		  IP_Face_LoRA_10   IP_Plus_LoRA_075  LoRA_075
IP_Face_LoRA_025  IP_Plus	    IP_Plus_LoRA_10   LoRA_10
IP_Face_LoRA_05   IP_Plus_LoRA_025  LoRA_025


In [5]:
MAIN_PROMPTS_FILE = "/kaggle/working/data/prompts.txt"         
EMOTION_PROMPTS_FILE = "/kaggle/working/data/emotion_prompts.txt" 

!mkdir "/kaggle/working/data"
with open(MAIN_PROMPTS_FILE, 'w') as f:
    f.write("""
a person wearing a space helmet
a person as a superhero
a person in a business suit holding a coffee
a person as a chef cooking pasta
a person skiing down a mountain
a person playing electric guitar on stage
a person as a pirate with a parrot
a person meditating in lotus pose
a person as an astronaut getting ready for the flight
a person riding a bicycle in the park
a person with a birthday cake
a person as a detective with a magnifying glass
a person as a knight in armor
a person surfing a big wave
a person as a painter at an easel
a person reading a book under a tree
a person as a wizard with a magic wand
a person taking a selfie with a dog
a person as a scientist with test tubes
a person dancing at a disco party
""")

with open(EMOTION_PROMPTS_FILE, 'w') as f:
    f.write("""
a person smiling happily
a person crying with tears
a person looking very angry
a person with a surprised expression
a person looking sad and thoughtful
a person laughing out loud
a person with a scared expression
a person looking disgusted
""")

# ========== ТЕКСТОВЫЕ ЛЕЙБЛЫ ДЛЯ CLIP-КЛАССИФИКАТОРА ЭМОЦИЙ ==========
# EMOTION_LABELS = [
#     "a happy smiling face",
#     "a sad crying face", 
#     "an angry furious face",
#     "a surprised shocked face",
#     "a neutral expressionless face",
# ]

In [19]:
import torch
import os
import numpy as np
import pandas as pd
from PIL import Image
from diffusers import StableDiffusionPipeline, DDIMScheduler
from peft import PeftModel
from insightface.app import FaceAnalysis
import open_clip
import torch.nn as nn
from tqdm import tqdm

# ========== НАСТРОЙКИ ==========
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16

LORA_SOURCE = "/kaggle/input/datasets/kolyagudkov/lora-diplom-weights"
REFERENCES_DIR = "/kaggle/input/datasets/kolyagudkov/ref2diplom"
MAIN_PROMPTS_FILE = "/kaggle/working/data/prompts.txt"
EMOTION_PROMPTS_FILE = "/kaggle/working/data/emotion_prompts.txt"

RESULTS_DIR = "/kaggle/working/results_scale_2"
METRICS_OUTPUT = "/kaggle/working/metrics_scale.csv"
BASE_MODEL = "SG161222/Realistic_Vision_V4.0_noVAE"

# os.makedirs(RESULTS_DIR, exist_ok=True)

# ========== ШАГ 1: СОЗДАНИЕ ОСЛАБЛЕННЫХ ВЕРСИЙ LORA ==========
print("=" * 60)
print("ШАГ 1: Создание версий LoRA с разным масштабом")
print("=" * 60)

LORA_PATHS = {
    0.0:  None,
    0.25: "/kaggle/working/lora_scale_025",
    0.5:  "/kaggle/working/lora_scale_050",
    0.75: "/kaggle/working/lora_scale_075",
    1.0:  "/kaggle/working/lora_scale_100",
}

# Загружаем базовую модель один раз
print("Загрузка базовой модели...")
base_pipe = StableDiffusionPipeline.from_pretrained(
    BASE_MODEL,
    torch_dtype=DTYPE,
    use_safetensors=True,
)
base_pipe = base_pipe.to(DEVICE)

for scale, path in LORA_PATHS.items():
    if scale == 0.0:
        continue
    if os.path.exists(path):
        print(f"  [SKIP] LoRA scale={scale:.2f} уже существует: {path}")
        continue
    
    print(f"  Создание LoRA scale={scale:.2f}...")
    pipe = StableDiffusionPipeline.from_pretrained(
        BASE_MODEL,
        torch_dtype=DTYPE,
        use_safetensors=True,
    )
    pipe.unet = PeftModel.from_pretrained(pipe.unet, LORA_SOURCE)
    
    # Масштабируем веса LoRA
    for name, param in pipe.unet.named_parameters():
        if "lora_" in name:
            param.data *= scale
    
    pipe.unet.save_pretrained(path)
    print(f"  [OK] Сохранено: {path}")
    del pipe
    torch.cuda.empty_cache()

del base_pipe
torch.cuda.empty_cache()

# ========== КОНФИГУРАЦИИ ==========
CONFIGS = [
    # Без LoRA
    {"name": "Base",              "lora_scale": 0.0,  "use_ip": False, "adapter_type": None},
    # {"name": "IP_Face",           "lora_scale": 0.0,  "use_ip": True,  "adapter_type": "face"},
    # {"name": "IP_Plus",           "lora_scale": 0.0,  "use_ip": True,  "adapter_type": "plus"},
    # LoRA scale = 0.25
    {"name": "LoRA_025",          "lora_scale": 0.25, "use_ip": False, "adapter_type": None},
    # {"name": "IP_Face_LoRA_025",  "lora_scale": 0.25, "use_ip": True,  "adapter_type": "face"},
    # {"name": "IP_Plus_LoRA_025",  "lora_scale": 0.25, "use_ip": True,  "adapter_type": "plus"},
    # LoRA scale = 0.5
    {"name": "LoRA_05",           "lora_scale": 0.5,  "use_ip": False, "adapter_type": None},
    # {"name": "IP_Face_LoRA_05",   "lora_scale": 0.5,  "use_ip": True,  "adapter_type": "face"},
    # {"name": "IP_Plus_LoRA_05",   "lora_scale": 0.5,  "use_ip": True,  "adapter_type": "plus"},
    # LoRA scale = 0.75
    {"name": "LoRA_075",          "lora_scale": 0.75, "use_ip": False, "adapter_type": None},
    # {"name": "IP_Face_LoRA_075",  "lora_scale": 0.75, "use_ip": True,  "adapter_type": "face"},
    # {"name": "IP_Plus_LoRA_075",  "lora_scale": 0.75, "use_ip": True,  "adapter_type": "plus"},
    # LoRA scale = 1.0
    {"name": "LoRA_10",           "lora_scale": 1.0,  "use_ip": False, "adapter_type": None},
    # {"name": "IP_Face_LoRA_10",   "lora_scale": 1.0,  "use_ip": True,  "adapter_type": "face"},
    # {"name": "IP_Plus_LoRA_10",   "lora_scale": 1.0,  "use_ip": True,  "adapter_type": "plus"},
]

# ========== ШАГ 2: ГЕНЕРАЦИЯ ==========
print("\n" + "=" * 60)
print("ШАГ 2: Генерация всех конфигураций")
print("=" * 60)

def load_pipeline(config):
    """Загружает пайплайн с нужной LoRA и IP-Adapter."""
    pipe = StableDiffusionPipeline.from_pretrained(
        BASE_MODEL,
        torch_dtype=DTYPE,
        use_safetensors=True,
    )
    pipe = pipe.to(DEVICE)
    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

    # LoRA
    lora_scale = config["lora_scale"]
    if lora_scale > 0.0:
        lora_path = LORA_PATHS[lora_scale]
        pipe.unet = PeftModel.from_pretrained(pipe.unet, lora_path)
        pipe.unet.set_adapter("default")
        print(f"  LoRA: scale={lora_scale:.2f}")

    # IP-Adapter
    if config["use_ip"]:
        weight_name = "ip-adapter-plus-face_sd15.safetensors" if config["adapter_type"] == "face" else "ip-adapter-plus_sd15.safetensors"
        pipe.load_ip_adapter(
            "h94/IP-Adapter",
            subfolder="models",
            weight_name=weight_name,
            image_encoder_folder="models/image_encoder",
        )
        pipe.set_ip_adapter_scale(0.7)
        print(f"  IP-Adapter: {config['adapter_type']}")

    return pipe


# Загружаем промпты
with open(MAIN_PROMPTS_FILE, "r") as f:
    main_prompts = [l.strip() for l in f if l.strip()]
with open(EMOTION_PROMPTS_FILE, "r") as f:
    emotion_prompts = [l.strip() for l in f if l.strip()]

# Загружаем референсы
ref_files = sorted([f for f in os.listdir(REFERENCES_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
ref_images = {}
for f in ref_files:
    name = os.path.splitext(f)[0]
    ref_images[name] = Image.open(os.path.join(REFERENCES_DIR, f)).convert("RGB")

prompt_sets = {"main": main_prompts, "emotion": emotion_prompts}

for config in CONFIGS:
    print(f"\n--- {config['name']} ---")
    config_dir = os.path.join(RESULTS_DIR, config["name"])
    existing = len(os.listdir(config_dir)) if os.path.isdir(config_dir) else 0
    expected = len(ref_images) * (len(main_prompts) + len(emotion_prompts))

    if existing >= expected:
        print(f"  [SKIP] Уже {existing}/{expected}")
        continue

    os.makedirs(config_dir, exist_ok=True)
    pipe = load_pipeline(config)

    for set_name, prompts in prompt_sets.items():
        for person_name, ref_image in ref_images.items():
            for i, prompt in enumerate(prompts):
                filename = f"{person_name}_{set_name}_{i:02d}.png"
                filepath = os.path.join(config_dir, filename)

                if os.path.exists(filepath):
                    continue

                try:
                    generator = torch.Generator(device=DEVICE).manual_seed(42)
                    kwargs = {
                        "prompt": prompt,
                        "negative_prompt": "low quality, blurry, distorted, ugly, bad anatomy",
                        "num_inference_steps": 30,
                        "guidance_scale": 7.0,
                        "generator": generator,
                        "height": 512,
                        "width": 512,
                    }
                    if config["use_ip"]:
                        kwargs["ip_adapter_image"] = ref_image

                    result = pipe(**kwargs)
                    result.images[0].save(filepath)
                except Exception as e:
                    print(f"  [ERROR] {filename}: {e}")

    del pipe
    torch.cuda.empty_cache()

print("\n✅ Генерация завершена!")

# ========== ШАГ 3: МЕТРИКИ ==========
print("\n" + "=" * 60)
print("ШАГ 3: Подсчёт метрик")
print("=" * 60)

def load_metric_models():
    print("Загрузка CLIP ViT-B-32...")
    clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
        "ViT-B-32", pretrained="laion2b_s34b_b79k"
    )
    clip_model = clip_model.to(DEVICE).eval()
    clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")

    print("Загрузка InsightFace...")
    face_app = FaceAnalysis(
        name="buffalo_l",
        providers=["CUDAExecutionProvider" if DEVICE == "cuda" else "CPUExecutionProvider"],
    )
    face_app.prepare(ctx_id=0 if DEVICE == "cuda" else -1)

    print("Загрузка Aesthetic Predictor...")
    aesthetic_clip, _, aesthetic_preprocess = open_clip.create_model_and_transforms(
        "ViT-L-14", pretrained="openai"
    )
    aesthetic_clip = aesthetic_clip.to(DEVICE).eval()

    aesthetic_model_path = "/kaggle/working/aesthetic_predictor.pth"
    if not os.path.exists(aesthetic_model_path):
        import urllib.request
        urllib.request.urlretrieve(
            "https://github.com/christophschuhmann/improved-aesthetic-predictor/raw/main/sac%2Blogos%2Bava1-l14-linearMSE.pth",
            aesthetic_model_path,
        )

    state_dict = torch.load(aesthetic_model_path, map_location=DEVICE)

    class AestheticMLP(nn.Module):
        def __init__(self):
            super().__init__()
            self.layers = nn.ModuleDict({})
            self.layers["0"] = nn.Linear(768, 1024)
            self.layers["2"] = nn.Linear(1024, 128)
            self.layers["4"] = nn.Linear(128, 64)
            self.layers["6"] = nn.Linear(64, 16)
            self.layers["7"] = nn.Linear(16, 1)
        def forward(self, x):
            x = nn.functional.relu(self.layers["0"](x))
            x = nn.functional.relu(self.layers["2"](x))
            x = nn.functional.relu(self.layers["4"](x))
            x = nn.functional.relu(self.layers["6"](x))
            x = self.layers["7"](x)
            return x.squeeze(-1)

    aesthetic_mlp = AestheticMLP().to(DEVICE)
    aesthetic_mlp.load_state_dict(state_dict, strict=True)
    aesthetic_mlp.eval()

    return clip_model, clip_preprocess, clip_tokenizer, face_app, aesthetic_clip, aesthetic_preprocess, aesthetic_mlp


clip_model, clip_preprocess, clip_tokenizer, face_app, aesthetic_clip, aesthetic_preprocess, aesthetic_mlp = load_metric_models()

all_prompts = main_prompts + emotion_prompts
text_emb_cache = {}
for prompt in all_prompts:
    text = clip_tokenizer([prompt]).to(DEVICE)
    with torch.no_grad():
        text_emb_cache[prompt] = clip_model.encode_text(text)
        text_emb_cache[prompt] = text_emb_cache[prompt] / text_emb_cache[prompt].norm(dim=-1, keepdim=True)

results = []

total = 0
for config in CONFIGS:
    config_path = os.path.join(RESULTS_DIR, config["name"])
    if os.path.isdir(config_path):
        total += len([f for f in os.listdir(config_path) if f.endswith(".png")])

pbar = tqdm(total=total, desc="Метрики")

for config in CONFIGS:
    config_path = os.path.join(RESULTS_DIR, config["name"])
    if not os.path.isdir(config_path):
        continue

    gen_files = sorted([f for f in os.listdir(config_path) if f.endswith(".png")])

    for gen_file in gen_files:
        parts = gen_file.replace(".png", "").split("_")
        person_name = parts[0]
        set_type = parts[1]
        prompt_idx = int(parts[2])

        prompt_text = main_prompts[prompt_idx] if set_type == "main" else emotion_prompts[prompt_idx]

        ref_path = None
        for ref_file in ref_files:
            if os.path.splitext(ref_file)[0] == person_name:
                ref_path = os.path.join(REFERENCES_DIR, ref_file)
                break

        if ref_path is None:
            pbar.update(1)
            continue

        gen_path = os.path.join(config_path, gen_file)

        # CLIP-I и CLIP-T
        ref_img = clip_preprocess(Image.open(ref_path).convert("RGB")).unsqueeze(0).to(DEVICE)
        gen_img = clip_preprocess(Image.open(gen_path).convert("RGB")).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            ref_emb = clip_model.encode_image(ref_img)
            gen_emb = clip_model.encode_image(gen_img)
            ref_emb = ref_emb / ref_emb.norm(dim=-1, keepdim=True)
            gen_emb = gen_emb / gen_emb.norm(dim=-1, keepdim=True)
            clip_i = float((ref_emb * gen_emb).sum(dim=-1).item())
            clip_t = float((gen_emb * text_emb_cache[prompt_text]).sum(dim=-1).item())

        # Face Similarity
        ref_faces = face_app.get(np.array(Image.open(ref_path).convert("RGB")))
        gen_faces = face_app.get(np.array(Image.open(gen_path).convert("RGB")))
        if len(ref_faces) > 0 and len(gen_faces) > 0:
            ref_face = max(ref_faces, key=lambda x: (x.bbox[2]-x.bbox[0])*(x.bbox[3]-x.bbox[1]))
            gen_face = max(gen_faces, key=lambda x: (x.bbox[2]-x.bbox[0])*(x.bbox[3]-x.bbox[1]))
            face_sim = float(np.dot(ref_face.normed_embedding, gen_face.normed_embedding))
        else:
            face_sim = float("nan")

        # Aesthetic
        aest_img = aesthetic_preprocess(Image.open(gen_path).convert("RGB")).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            features = aesthetic_clip.encode_image(aest_img)
            features = features / features.norm(dim=-1, keepdim=True)
            aesthetic = float(aesthetic_mlp(features).item())

        results.append({
            "config": config["name"],
            "lora_scale": config["lora_scale"],
            "use_ip": config["use_ip"],
            "adapter_type": config["adapter_type"],
            "person": person_name,
            "set": set_type,
            "prompt": prompt_text,
            "CLIP_I": round(clip_i, 4),
            "CLIP_T": round(clip_t, 4),
            "Face_Similarity": round(face_sim, 4) if not np.isnan(face_sim) else "N/A",
            "Aesthetic_Score": round(aesthetic, 2),
        })

        pbar.update(1)

pbar.close()

df = pd.DataFrame(results)
df.to_csv(METRICS_OUTPUT, index=False)

# ========== ТАБЛИЦЫ ==========
df_clean = df.replace("N/A", float("nan"))
df_clean["Face_Similarity"] = pd.to_numeric(df_clean["Face_Similarity"], errors="coerce")

# ТАБЛИЦА 1: Все конфигурации
summary_all = (
    df_clean.groupby("config")
    .agg(
        lora_scale=("lora_scale", "first"),
        use_ip=("use_ip", "first"),
        adapter=("adapter_type", "first"),
        CLIP_I=("CLIP_I", "mean"),
        CLIP_T=("CLIP_T", "mean"),
        Face_Sim=("Face_Similarity", "mean"),
        Aesthetic=("Aesthetic_Score", "mean"),
        Face_Detect=("Face_Similarity", lambda x: x.notna().sum()),
        Total=("CLIP_I", "count"),
    )
    .round(4)
)
summary_all["Face_Det%"] = (summary_all["Face_Detect"] / summary_all["Total"] * 100).round(1)

print(f"\n{'='*100}")
print("ТАБЛИЦА 1: ВСЕ КОНФИГУРАЦИИ")
print(f"{'='*100}")
print(f"{'Конфигурация':<22} {'Scale':>5} {'IP':>6} {'CLIP-I':>8} {'CLIP-T':>8} {'Face':>8} {'Aesth':>6} {'Det%':>6}")
print("-"*100)
for _, row in summary_all.iterrows():
    ip_label = row["adapter"] if row["use_ip"] else "-"
    print(f"{row.name:<22} {row['lora_scale']:.2f} {ip_label:>6} "
          f"{row['CLIP_I']:.3f} {row['CLIP_T']:.3f} {row['Face_Sim']:.3f} "
          f"{row['Aesthetic']:.2f} {row['Face_Det%']:.1f}%")

# ТАБЛИЦА 2: Влияние scale (только IP_Face)
face_configs = [c for c in CONFIGS if c["use_ip"] and c["adapter_type"] == "face"]
face_subset = df_clean[df_clean["config"].isin([c["name"] for c in face_configs])]
face_summary = (
    face_subset.groupby("lora_scale")
    .agg(
        CLIP_I=("CLIP_I", "mean"),
        CLIP_T=("CLIP_T", "mean"),
        Face_Sim=("Face_Similarity", "mean"),
        Aesthetic=("Aesthetic_Score", "mean"),
    )
    .round(4)
)

print(f"\n{'='*70}")
print("ТАБЛИЦА 2: IP_Face + LoRA — ВЛИЯНИЕ SCALE")
print(f"{'='*70}")
print(f"{'Scale':<10} {'CLIP-I':>8} {'CLIP-T':>8} {'Face Sim':>8} {'Aesthetic':>6}")
print("-"*50)
for scale, row in face_summary.iterrows():
    print(f"{scale:.2f}      {row['CLIP_I']:.3f}   {row['CLIP_T']:.3f}   {row['Face_Sim']:.3f}   {row['Aesthetic']:.2f}")

# ТАБЛИЦА 3: По сетам
print(f"\n{'='*90}")
print("ТАБЛИЦА 3: РАЗДЕЛЬНО ПО СЕТАМ")
print(f"{'='*90}")
for set_name, set_label in [("main", "Основные промпты"), ("emotion", "Эмоции")]:
    subset = df_clean[df_clean["set"] == set_name]
    summary_set = (
        subset.groupby("config")
        .agg(CLIP_I=("CLIP_I", "mean"), CLIP_T=("CLIP_T", "mean"),
             Face_Sim=("Face_Similarity", "mean"), Aesthetic=("Aesthetic_Score", "mean"))
        .round(4)
    )
    print(f"\n--- {set_label} ---")
    print(f"{'Конфигурация':<22} {'CLIP-I':>8} {'CLIP-T':>8} {'Face':>8} {'Aesthetic':>6}")
    print("-"*65)
    for cfg in CONFIGS:
        name = cfg["name"]
        if name in summary_set.index:
            row = summary_set.loc[name]
            print(f"{name:<22} {row['CLIP_I']:.3f}   {row['CLIP_T']:.3f}   {row['Face_Sim']:.3f}   {row['Aesthetic']:.2f}")

print(f"\n✅ Детальные метрики: {METRICS_OUTPUT}")
print("Готово!")

ШАГ 1: Создание версий LoRA с разным масштабом
Загрузка базовой модели...


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--SG161222--Realistic_Vision_V4.0_noVAE/snapshots/1685907c0283c7278ba26c5fe561506f564b48d3/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [SKIP] LoRA scale=0.25 уже существует: /kaggle/working/lora_scale_025
  [SKIP] LoRA scale=0.50 уже существует: /kaggle/working/lora_scale_050
  [SKIP] LoRA scale=0.75 уже существует: /kaggle/working/lora_scale_075
  [SKIP] LoRA scale=1.00 уже существует: /kaggle/working/lora_scale_100

ШАГ 2: Генерация всех конфигураций

--- Base ---
  [SKIP] Уже 140/140

--- LoRA_025 ---
  [SKIP] Уже 140/140

--- LoRA_05 ---
  [SKIP] Уже 140/140

--- LoRA_075 ---
  [SKIP] Уже 140/140

--- LoRA_10 ---
  [SKIP] Уже 140/140

✅ Генерация завершена!

ШАГ 3: Подсчёт метрик
Загрузка CLIP ViT-B-32...


...

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: [(128, 128), (640, 640)

open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(

Метрики: 100%|██████████| 700/700 [15:56<00:00,  1.37s/it]


ТАБЛИЦА 1: ВСЕ КОНФИГУРАЦИИ
Конфигурация           Scale     IP   CLIP-I   CLIP-T     Face  Aesth   Det%
----------------------------------------------------------------------------------------------------
Base                   0.00      - 0.309 0.327 -0.000 4.04 78.6%
LoRA_025               0.25      - 0.305 0.323 0.010 4.03 78.6%
LoRA_05                0.50      - 0.302 0.325 0.005 4.02 78.6%
LoRA_075               0.75      - 0.300 0.318 -0.001 3.96 89.3%
LoRA_10                1.00      - 0.293 0.231 0.005 3.78 53.6%

ТАБЛИЦА 2: IP_Face + LoRA — ВЛИЯНИЕ SCALE
Scale        CLIP-I   CLIP-T Face Sim Aesthetic
--------------------------------------------------

ТАБЛИЦА 3: РАЗДЕЛЬНО ПО СЕТАМ

--- Основные промпты ---
Конфигурация             CLIP-I   CLIP-T     Face Aesthetic
-----------------------------------------------------------------
Base                   0.262   0.325   0.011   4.06
LoRA_025               0.267   0.328   0.018   4.05
LoRA_05                0.267   0.330   0.0


/tmp/ipykernel_57/1632647518.py:358: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_clean = df.replace("N/A", float("nan"))


In [4]:
# !pip install insightface onnxruntime open-clip-torch